# Boosting：串行训练、逐步纠错
> 难度标签：中级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：07_集成学习 | 源文件：Boosting.py | 核心功能：串行集成、残差拟合、偏差降低

## 概述

Boosting 与 Bagging 的核心区别：基模型**串行**训练，每个新模型专注于修正前一个模型的错误。核心价值：**降低偏差**。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier
# --- 导入库 ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, r2_score

## 数学原理

### 1. Boosting 的基本思想

Boosting 将多个**弱学习器**（性能略优于随机猜测）串行组合为**强学习器**。核心策略：每一轮重点关注前一轮预测错误的样本。

### 2. 加法模型（Additive Model）

Boosting 构建的集成模型是加法展开：

$$F_M(x) = \sum_{m=1}^{M} \alpha_m f_m(x)$$

其中 $f_m$ 是第 $m$ 轮的基学习器，$\alpha_m$ 是其权重，$M$ 是总轮数。

### 3. 梯度提升（Gradient Boosting）

Gradient Boosting 的核心思想：每一轮拟合**损失函数的负梯度**（伪残差）。

设损失函数为 $L(y, F(x))$，第 $m$ 轮的伪残差：

$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}, \quad i=1,\ldots,n$$

**二分类**（对数损失）：$L = -[y\log p + (1-y)\log(1-p)]$，伪残差为 $r_{im} = y_i - p_{m-1}(x_i)$。

**回归**（MSE）：$L = \frac{1}{2}(y-F)^2$，伪残差为 $r_{im} = y_i - F_{m-1}(x_i)$（即当前残差）。

### 4. 基学习器拟合伪残差

第 $m$ 轮用弱学习器（如深度为 $J$ 的决策树）拟合伪残差：

$$f_m(x) = \arg\min_f \sum_{i=1}^{n} (r_{im} - f(x_i))^2$$

树将输入空间划分为 $R_{jm}$ 个区域（叶节点），每个区域的输出值：

$$\gamma_{jm} = \arg\min_\gamma \sum_{x_i \in R_{jm}} L(y_i, F_{m-1}(x_i) + \gamma)$$

### 5. 模型更新

$$F_m(x) = F_{m-1}(x) + \nu \cdot \sum_{j=1}^{J} \gamma_{jm} \mathbb{I}[x \in R_{jm}]$$

其中 $\nu$ 是学习率（`learning_rate`），控制每步的步长。

### 6. 学习率与迭代次数的权衡

$$F_M(x) = \sum_{m=1}^{M} \nu \cdot f_m(x)$$

- $\nu$ 小 → 每步贡献小 → 需要更多 $M$ → 训练更慢，但泛化通常更好
- $\nu$ 大 → 每步贡献大 → 收敛快 → 容易过拟合

代码中实验了 $(0.3, 50), (0.1, 100), (0.05, 200), (0.01, 500)$ 的组合。

### 7. 随机梯度提升（Stochastic GB）

引入 `subsample` 参数，每轮只使用 $80\%$ 的随机子集训练：

$$r_{im} = -\frac{\partial L}{\partial F}\bigg|_{F=F_{m-1}}, \quad i \in S_m, \quad |S_m| = n \cdot \text{subsample}$$

类似于 Bagging 的随机性，降低基学习器之间的相关性，减少过拟合。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `GradientBoostingClassifier(n_estimators=100)` | 加法模型 $F_{100} = \sum_{m=1}^{100}\alpha_m f_m$ |
| `learning_rate=0.1` | 步长 $\nu=0.1$，控制每轮贡献 |
| `max_depth=3` | 基学习器为深度 $J=3$ 的决策树 |
| `subsample=0.8` | 随机梯度提升，每轮用 $80\%$ 样本 |
| `feature_importances_` | 基于特征在所有树中分裂带来的损失减少之和 |
| `max_depth=1`（弱学习器） | 决策树桩，Boosting 的典型弱学习器 |

### 1. 生成数据

生成合成数据集，用于演示算法的核心行为。

In [ ]:
X, y = make_classification(n_samples=500, n_features=10, n_informative=5,
                           n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 2. 基准：单个弱学习器

运行 2. 基准：单个弱学习器 的代码，观察算法在该环节的行为。

In [ ]:
dt = DecisionTreeClassifier(max_depth=1, random_state=42)  # 决策树桩
dt.fit(X_train, y_train)
print(f"=== 单个决策树桩: 测试准确率={dt.score(X_test, y_test):.4f} ===")

### 3. GradientBoosting 分类

在分类任务上训练模型并评估性能。

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,  # 随机梯度提升（Stochastic GB）
# --- 赋值/配置 ---
    random_state=42,
)
gb.fit(X_train, y_train)

print(f"\n=== GradientBoosting ===")
print(f"训练准确率: {gb.score(X_train, y_train):.4f}")
print(f"测试准确率: {gb.score(X_test, y_test):.4f}")
print(f"特征重要性: {gb.feature_importances_.round(4)}")

### 4. n_estimators 影响

运行 4. n_estimators 影响 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== n_estimators 对比 ===")
for n in [10, 25, 50, 100, 200, 500]:
    gb_n = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, random_state=42)
    gb_n.fit(X_train, y_train)
    print(f"  n={n:>3}: 测试准确率={gb_n.score(X_test, y_test):.4f}")

### 5. learning_rate vs n_estimators

运行 5. learning_rate vs n_estimators 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== learning_rate vs n_estimators ===")
print("learning_rate 越小，需要更多轮次，但泛化通常更好")
for lr, n in [(0.3, 50), (0.1, 100), (0.05, 200), (0.01, 500)]:
    gb_lr = GradientBoostingClassifier(n_estimators=n, learning_rate=lr, random_state=42)
    gb_lr.fit(X_train, y_train)
# --- 输出结果 ---
    print(f"  lr={lr}, n={n:>3}: 测试准确率={gb_lr.score(X_test, y_test):.4f}")

### 6. subsample（随机梯度提升）

运行 6. subsample（随机梯度提升） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== subsample 对比 ===")
for ss in [0.5, 0.7, 0.8, 1.0]:
    gb_s = GradientBoostingClassifier(n_estimators=100, subsample=ss, random_state=42)
    gb_s.fit(X_train, y_train)
    print(f"  subsample={ss}: 测试准确率={gb_s.score(X_test, y_test):.4f}")

### 7. Boosting 回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print("\n=== GradientBoosting 回归 ===")
from sklearn.datasets import make_regression
X_r, y_r = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
gb_r = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
# --- 训练模型 ---
gb_r.fit(X_tr, y_tr)
print(f"R²: {gb_r.score(X_te, y_te):.4f}")

### 8. Boosting 原理

运行 8. Boosting 原理 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Boosting 原理 ===")
print("1. 初始化弱学习器 F_0(x)")
print("2. 对每轮 m = 1, 2, ..., M:")
print("   a. 计算残差（负梯度）r_m = -∂L/∂F")
print("   b. 训练新弱学习器拟合残差")
# --- 输出结果 ---
print("   c. 更新 F_m = F_{m-1} + η × h_m(x)")
print("3. 最终模型: F_M(x) = Σ η × h_m(x)")

print("\n=== Boosting vs Bagging ===")
print("Bagging: 并行训练，降低方差，适合不稳定模型")
print("Boosting: 串行训练，降低偏差，适合弱学习器")
print("Boosting 更容易过拟合（需限制 n_estimators/learning_rate）")

print("\n=== Boosting 要点 ===")
print("- learning_rate × n_estimators 的权衡：小 lr + 大 n 通常更好")
print("- max_depth 通常较小（3~6），基学习器是弱学习器")
print("- subsample 引入随机性，防止过拟合")
print("- 对异常值敏感（残差可能被异常值主导）")

## 关键代码解释

```python
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
```

`learning_rate` 控制每棵树的贡献权重（缩小步长，防止过拟合）。

## 注意事项

1. 不能并行（串行依赖）
2. 容易过拟合（需控制 n_estimators 和 learning_rate）
3. learning_rate 和 n_estimators 通常反向调节

## 总结与延伸

以上代码展示了 **Boosting** 的完整流程。

**解读要点**：
- 对比 **单模型 vs 集成模型** 的性能提升
- 关注 **特征重要性** 排名
- 观察 OOB 分数（随机森林）或 early stopping 轮次（XGBoost）

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **XGBoost/LightGBM/CatBoost**：工业级 Boosting 实现
- **AdaBoost**：最早的 Boosting 算法
- **学习率调度**：逐步降低学习率
